In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

In [23]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [24]:
NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DAILY_DATA_DIR = PROJECT_ROOT / "data" / "daily"
MINUTE_DATA_DIR = PROJECT_ROOT / "data" / "minute"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RESEARCH_DIR = PROJECT_ROOT / "research"

PROCESSED_DATA_DIR = OUTPUTS_DIR / "processed_data"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Daily data directory:", DAILY_DATA_DIR)
print("Minute data directory:", MINUTE_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

Notebook directory: /Users/kushagr/Desktop/astra-assignment/notebooks
Project root: /Users/kushagr/Desktop/astra-assignment
Daily data directory: /Users/kushagr/Desktop/astra-assignment/data/daily
Minute data directory: /Users/kushagr/Desktop/astra-assignment/data/minute
Processed data directory: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data


In [25]:
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert DAILY_DATA_DIR.exists(), f"Daily data directory not found: {DAILY_DATA_DIR}"
assert MINUTE_DATA_DIR.exists(), f"Minute data directory not found: {MINUTE_DATA_DIR}"

print("All required directories were found.")

All required directories were found.


In [26]:
daily_files = sorted(DAILY_DATA_DIR.glob("*.parquet"))

print("Number of daily parquet files:", len(daily_files))

for file_path in daily_files[:10]:
    print(file_path.name)

Number of daily parquet files: 208
360ONE.parquet
ABB.parquet
ABCAPITAL.parquet
ADANIENSOL.parquet
ADANIENT.parquet
ADANIGREEN.parquet
ADANIPORTS.parquet
ADANIPOWER.parquet
ALKEM.parquet
AMBER.parquet


In [27]:
EXPECTED_SYMBOL_COUNT = 208

if len(daily_files) == EXPECTED_SYMBOL_COUNT:
    print("Daily file count matches the assignment.")
else:
    print(
        f"Warning: expected {EXPECTED_SYMBOL_COUNT} files, "
        f"but found {len(daily_files)}."
    )

Daily file count matches the assignment.


In [28]:
sample_daily_file = daily_files[0]
sample_symbol = sample_daily_file.stem

sample_daily_df = pd.read_parquet(sample_daily_file)

print("Sample file:", sample_daily_file.name)
print("Symbol:", sample_symbol)
print("Shape:", sample_daily_df.shape)

print(sample_daily_df.head())
print("\n" * 2)
print(sample_daily_df.info())


Sample file: 360ONE.parquet
Symbol: 360ONE
Shape: (1511, 6)
         date     open     high      low    close  volume
0  2020-06-01 209.1500 230.9500 209.1500 219.2500  185884
1  2020-06-02 224.9500 228.7500 218.9000 224.6500   22368
2  2020-06-03 228.7500 230.0000 221.2500 227.7000   34560
3  2020-06-04 233.7500 247.5000 228.7500 239.8500   77532
4  2020-06-05 244.0500 253.0500 236.5000 250.1500   22376



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    1511 non-null   object 
 1   open    1511 non-null   float64
 2   high    1511 non-null   float64
 3   low     1511 non-null   float64
 4   close   1511 non-null   float64
 5   volume  1511 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 71.0+ KB
None


In [29]:
EXPECTED_DAILY_COLUMNS = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

print("Columns found:", sample_daily_df.columns.tolist())

missing_columns = set(EXPECTED_DAILY_COLUMNS) - set(sample_daily_df.columns)

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("Expected daily columns are present.")

Columns found: ['date', 'open', 'high', 'low', 'close', 'volume']
Expected daily columns are present.


In [30]:
def load_daily_file(file_path: Path) -> pd.DataFrame:
    """
    Load one stock's daily parquet file and add its symbol.

    Parameters
    ----------
    file_path : Path
        Path to the parquet file.

    Returns
    -------
    pd.DataFrame
        Clean daily OHLCV data with a symbol column.
    """
    df = pd.read_parquet(file_path).copy()

    df.columns = [str(column).strip().lower() 
                  for column in df.columns
                  ]

    required_columns = {"date", "open", "high", "low", "close", "volume",}

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    df = df[["date","open","high","low","close","volume",]].copy()

    df["date"] = pd.to_datetime(df["date"], errors="raise",)

    df["symbol"] = file_path.stem

    return df

In [31]:
test_daily_df = load_daily_file(sample_daily_file)

print(test_daily_df.shape)
test_daily_df.head()

(1511, 7)


,date,open,high,low,close,volume,symbol
0,2020-06-01,209.1500,230.9500,209.1500,219.2500,185884,360ONE
1,2020-06-02,224.9500,228.7500,218.9000,224.6500,22368,360ONE
2,2020-06-03,228.7500,230.0000,221.2500,227.7000,34560,360ONE
3,2020-06-04,233.7500,247.5000,228.7500,239.8500,77532,360ONE
4,2020-06-05,244.0500,253.0500,236.5000,250.1500,22376,360ONE


In [32]:
daily_frames = []

for file_path in daily_files:
    stock_daily_df = load_daily_file(file_path)
    daily_frames.append(stock_daily_df)

daily_df = pd.concat(daily_frames,axis=0, ignore_index=True,)

daily_df = daily_df.sort_values(["symbol", "date"]).reset_index(drop=True)

daily_df = daily_df[["date", "symbol", "open", "high", "low", "close", "volume", ]]

print("Combined daily dataset shape:", daily_df.shape)
daily_df.head()

Combined daily dataset shape: (301483, 7)


,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


In [33]:
print("Rows:", len(daily_df))
print("Symbols:", daily_df["symbol"].nunique())
print("Trading dates:", daily_df["date"].nunique())
print("Date range:", daily_df["date"].min(), "to", daily_df["date"].max(),)

Rows: 301483
Symbols: 208
Trading dates: 1511
Date range: 2020-06-01 00:00:00 to 2026-06-30 00:00:00


In [34]:
duplicate_mask = daily_df.duplicated(subset=["symbol", "date"],keep=False,)

duplicate_count = int(duplicate_mask.sum())

print("Duplicate symbol-date rows:", duplicate_count)

if duplicate_count > 0:
    display(
        daily_df.loc[duplicate_mask]
        .sort_values(["symbol", "date"])
        .head(20)
    )

Duplicate symbol-date rows: 0


In [35]:
missing_summary = daily_df.isna().sum().to_frame(name="missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"]/ len(daily_df)* 100)
missing_summary

,missing_count,missing_pct
date,0,0.0000
symbol,0,0.0000
open,0,0.0000
high,0,0.0000
low,0,0.0000
close,0,0.0000
volume,0,0.0000


In [36]:
quality_checks = pd.Series(
    {
        "non_positive_open": (daily_df["open"] <= 0).sum(),
        "non_positive_high": (daily_df["high"] <= 0).sum(),
        "non_positive_low": (daily_df["low"] <= 0).sum(),
        "non_positive_close": (daily_df["close"] <= 0).sum(),
        "negative_volume": (daily_df["volume"] < 0).sum(),
        "high_below_low": (
            daily_df["high"] < daily_df["low"]
        ).sum(),
        "high_below_open": (
            daily_df["high"] < daily_df["open"]
        ).sum(),
        "high_below_close": (
            daily_df["high"] < daily_df["close"]
        ).sum(),
        "low_above_open": (
            daily_df["low"] > daily_df["open"]
        ).sum(),
        "low_above_close": (
            daily_df["low"] > daily_df["close"]
        ).sum(),
    },
    name="count",
)

quality_checks

non_positive_open     0
non_positive_high     0
non_positive_low      0
non_positive_close    0
negative_volume       0
high_below_low        0
high_below_open       0
high_below_close      0
low_above_open        0
low_above_close       0
Name: count, dtype: int64

In [37]:
rows_per_symbol = (daily_df.groupby("symbol").size().sort_values())

rows_per_symbol.describe()

count     208.0000
mean    1,449.4375
std       210.6202
min       378.0000
25%     1,511.0000
50%     1,511.0000
75%     1,511.0000
max     1,511.0000
dtype: float64

In [38]:
rows_per_symbol.head(20)

symbol
VMM            378
SWIGGY         401
WAAREEENER     413
HYUNDAI        417
PREMIERENE     451
IREDA          640
JIOFIN         708
MANKIND        780
KFINTECH       866
KAYNES         893
DELHIVERY     1017
LICI          1022
PAYTM         1143
POLICYBZR     1146
NYKAA         1149
ETERNAL       1223
SONACOMS      1243
LODHA         1289
KALYANKJIL    1302
IRFC          1341
dtype: int64

In [39]:
symbols_per_date = (daily_df.groupby("date")["symbol"].nunique().sort_values())

symbols_per_date.head(10)

date
2020-06-01    184
2020-07-15    184
2020-07-16    184
2020-07-17    184
2020-07-20    184
2020-07-22    184
2020-07-23    184
2020-07-24    184
2020-07-27    184
2020-07-28    184
Name: symbol, dtype: int64

In [40]:
daily_panel_path = (PROCESSED_DATA_DIR / "daily_panel_raw.parquet")

daily_df.to_parquet(daily_panel_path, index=False,)

print("Saved combined daily panel to:")
print(daily_panel_path)

Saved combined daily panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_panel_raw.parquet


In [41]:
daily_df.head()

,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


# Target Construction


In [42]:
target_df = daily_df.copy()

target_df = target_df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(target_df.shape)
target_df.head()

(301483, 7)


,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


In [43]:
symbol_groups = target_df.groupby("symbol",sort=False,)

target_df["target_date"] = (symbol_groups["date"].shift(-1))

target_df["next_open"] = (symbol_groups["open"].shift(-1))

In [44]:
target_df["actual_return_pct"] = ((target_df["next_open"]/ target_df["close"])- 1) * 100

target_df["actual_direction"] = np.where(target_df["actual_return_pct"] >= 0,1,-1,)

target_df["actual_magnitude_pct"] = (target_df["actual_return_pct"].abs())

In [45]:
target_df["calendar_gap_days"] = (target_df["target_date"]- target_df["date"]).dt.days

In [46]:
target_columns = ["date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]

target_df[target_columns].head(5)

,date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,2020-06-02,360ONE,219.2500,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,2020-06-03,360ONE,224.6500,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,2020-06-04,360ONE,227.7000,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,2020-06-05,360ONE,239.8500,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,2020-06-08,360ONE,250.1500,251.7500,0.6396,1,0.6396,3.0000


In [47]:
sample_symbol = "RELIANCE"

target_df.loc[target_df["symbol"] == sample_symbol, target_columns,].head(5)

,date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
243141,2020-06-01,2020-06-02,RELIANCE,724.6000,727.3000,0.3726,1,0.3726,1.0000
243142,2020-06-02,2020-06-03,RELIANCE,731.9000,736.3500,0.6080,1,0.6080,1.0000
243143,2020-06-03,2020-06-04,RELIANCE,734.7500,735.9000,0.1565,1,0.1565,1.0000
243144,2020-06-04,2020-06-05,RELIANCE,752.9000,760.2000,0.9696,1,0.9696,1.0000
243145,2020-06-05,2020-06-08,RELIANCE,753.8500,771.3000,2.3148,1,2.3148,3.0000


In [48]:
valid_target_mask = target_df["target_date"].notna()

invalid_target_dates = (target_df.loc[valid_target_mask, "target_date"] <= target_df.loc[valid_target_mask, "date"]).sum()

print("Rows where target_date is not after date:",invalid_target_dates,)

Rows where target_date is not after date: 0


In [49]:
target_df["actual_direction"].value_counts(dropna=False)

actual_direction
 1    214039
-1     87444
Name: count, dtype: int64

In [50]:
missing_target_mask = target_df["actual_return_pct"].isna()

target_df.loc[missing_target_mask,"actual_direction",] = np.nan

In [51]:
target_df["actual_direction"].value_counts(dropna=False)

actual_direction
1.0000     214039
-1.0000     87236
NaN           208
Name: count, dtype: int64

In [52]:
negative_magnitude_count = (target_df["actual_magnitude_pct"] < 0).sum()

print("Negative magnitude rows:",negative_magnitude_count,)

Negative magnitude rows: 0


In [53]:
target_df["calendar_gap_days"].value_counts(dropna=False).sort_index()

calendar_gap_days
1.0000      228739
2.0000       10085
3.0000       55846
4.0000        6408
5.0000         196
112.0000         1
NaN            208
Name: count, dtype: int64

In [54]:
missing_target_summary = target_df[["target_date","next_open","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().sum()

missing_target_summary

target_date             208
next_open               208
actual_return_pct       208
actual_direction        208
actual_magnitude_pct    208
dtype: int64

In [55]:
daily_with_targets_full = target_df.copy()


daily_with_targets = (
    target_df
    .dropna(
        subset=[
            "target_date",
            "next_open",
            "actual_return_pct",
            "actual_direction",
            "actual_magnitude_pct",
        ]
    )
    .copy()
)

daily_with_targets["actual_direction"] = (daily_with_targets["actual_direction"].astype("int8"))

print("Rows before dropping missing targets:",len(target_df),)

print("Rows after dropping missing targets:",len(daily_with_targets),)

print("Rows removed:",len(target_df) - len(daily_with_targets),)

Rows before dropping missing targets: 301483
Rows after dropping missing targets: 301275
Rows removed: 208


In [56]:
daily_with_targets = daily_with_targets.rename(columns={"date": "pred_date"})


daily_with_targets[["pred_date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]].head()

,pred_date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,2020-06-02,360ONE,219.2500,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,2020-06-03,360ONE,224.6500,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,2020-06-04,360ONE,227.7000,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,2020-06-05,360ONE,239.8500,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,2020-06-08,360ONE,250.1500,251.7500,0.6396,1,0.6396,3.0000


In [57]:
assert not daily_with_targets[["pred_date","target_date","symbol","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().any().any()

assert (daily_with_targets["target_date"] > daily_with_targets["pred_date"]).all()

assert daily_with_targets["actual_direction"].isin([-1, 1]).all()

assert (daily_with_targets["actual_magnitude_pct"] >= 0).all()

assert np.allclose(daily_with_targets["actual_magnitude_pct"],daily_with_targets["actual_return_pct"].abs(),)

print("All target-construction checks passed.")

All target-construction checks passed.


In [58]:
target_panel_path = (PROCESSED_DATA_DIR/ "daily_panel_with_targets_v1.parquet")

daily_with_targets.to_parquet(target_panel_path,index=False,)

print("Saved target panel to:", target_panel_path)

Saved target panel to: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_panel_with_targets_v1.parquet


In [59]:
daily_with_targets.head()

,pred_date,symbol,open,high,low,close,volume,target_date,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884,2020-06-02,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368,2020-06-03,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560,2020-06-04,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532,2020-06-05,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376,2020-06-08,251.7500,0.6396,1,0.6396,3.0000




### Daily Feature Engineering — Version 1

Daily features are calculated separately within each stock.

All features for prediction date `T` use only information available by the close of `T`.

Historical overnight features are shifted by one row because `actual_return_pct`
for row `T` is only realised at the open of `target_date`.

In [60]:
daily_features_df = daily_with_targets.copy()

daily_features_df = daily_features_df.sort_values(["symbol", "pred_date"]).reset_index(drop=True)

print("Starting shape:", daily_features_df.shape)
daily_features_df.head()

Starting shape: (301275, 13)


,pred_date,symbol,open,high,low,close,volume,target_date,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884,2020-06-02,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368,2020-06-03,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560,2020-06-04,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532,2020-06-05,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376,2020-06-08,251.7500,0.6396,1,0.6396,3.0000


In [61]:
symbol_groups = daily_features_df.groupby("symbol",sort=False,group_keys=False,)

In [62]:
# The current row's actual_return_pct belongs to the future target.
# Shift by one so the feature contains only previously realised gaps.

daily_features_df["lagged_overnight_return_1d"] = (symbol_groups["actual_return_pct"].shift(1))

daily_features_df["gap_mean_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["overnight_std_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

daily_features_df["gap_positive_fraction_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: (series.ge(0).where(series.notna()).rolling(window=20,min_periods=20,).mean())))

In [63]:
overnight_feature_columns = ["pred_date","target_date","symbol","actual_return_pct","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",overnight_feature_columns,].head(25)

,pred_date,target_date,symbol,actual_return_pct,lagged_overnight_return_1d,gap_mean_20d,overnight_std_20d,gap_positive_fraction_20d
242974,2020-06-01,2020-06-02,RELIANCE,0.3726,NaN,NaN,NaN,NaN
242975,2020-06-02,2020-06-03,RELIANCE,0.6080,0.3726,NaN,NaN,NaN
242976,2020-06-03,2020-06-04,RELIANCE,0.1565,0.6080,NaN,NaN,NaN
242977,2020-06-04,2020-06-05,RELIANCE,0.9696,0.1565,NaN,NaN,NaN
242978,2020-06-05,2020-06-08,RELIANCE,2.3148,0.9696,NaN,NaN,NaN
242979,2020-06-08,2020-06-09,RELIANCE,-0.5816,2.3148,NaN,NaN,NaN
242980,2020-06-09,2020-06-10,RELIANCE,0.3890,-0.5816,NaN,NaN,NaN
242981,2020-06-10,2020-06-11,RELIANCE,-0.3937,0.3890,NaN,NaN,NaN
242982,2020-06-11,2020-06-12,RELIANCE,-2.4560,-0.3937,NaN,NaN,NaN
242983,2020-06-12,2020-06-15,RELIANCE,-1.4923,-2.4560,NaN,NaN,NaN


In [64]:
daily_features_df["return_1d"] = (symbol_groups["close"].pct_change(1) * 100)

daily_features_df["return_5d"] = (symbol_groups["close"].pct_change(5) * 100)

daily_features_df["return_20d"] = (symbol_groups["close"].pct_change(20) * 100)

In [65]:
daily_features_df["daily_volatility_5d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=5,min_periods=5,).std()))

daily_features_df["daily_volatility_20d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

In [67]:
daily_features_df["volume_mean_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).mean())))

daily_features_df["volume_std_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).std())))

daily_features_df["volume_zscore_20d"] = ((daily_features_df["volume"]- daily_features_df["volume_mean_20d_lagged"])/ daily_features_df["volume_std_20d_lagged"])

In [69]:
daily_features_df["volume_mean_5d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=5,min_periods=5,).mean()))

daily_features_df["volume_mean_20d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["volume_trend_5d_20d"] = (daily_features_df["volume_mean_5d"] / daily_features_df["volume_mean_20d"])

In [70]:
daily_features_df["day_of_week"] = (daily_features_df["pred_date"].dt.dayofweek)

In [71]:
daily_feature_columns = ["lagged_overnight_return_1d","return_1d","return_5d","return_20d","daily_volatility_20d","overnight_std_20d","daily_volatility_5d","gap_mean_20d","gap_positive_fraction_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week",]

print("Number of unique Version 1 daily features:", len(daily_feature_columns))

Number of unique Version 1 daily features: 13


In [72]:
daily_feature_missingness = daily_features_df[daily_feature_columns].isna().sum().to_frame("missing_count")

daily_feature_missingness["missing_pct"] = (daily_feature_missingness["missing_count"] / len(daily_features_df) * 100)

daily_feature_missingness.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
return_20d,4160,1.3808
daily_volatility_20d,4160,1.3808
overnight_std_20d,4160,1.3808
gap_mean_20d,4160,1.3808
gap_positive_fraction_20d,4160,1.3808
volume_zscore_20d,4160,1.3808
volume_trend_5d_20d,3952,1.3118
return_5d,1040,0.3452
daily_volatility_5d,1040,0.3452
lagged_overnight_return_1d,208,0.0690


In [73]:
daily_feature_infinite_counts = pd.Series({column: np.isinf(daily_features_df[column]).sum() for column in daily_feature_columns},name="infinite_count",)

daily_feature_infinite_counts

lagged_overnight_return_1d    0
return_1d                     0
return_5d                     0
return_20d                    0
daily_volatility_20d          0
overnight_std_20d             0
daily_volatility_5d           0
gap_mean_20d                  0
gap_positive_fraction_20d     0
volume_zscore_20d             0
volume_trend_5d_20d           0
calendar_gap_days             0
day_of_week                   0
Name: infinite_count, dtype: int64

In [74]:
daily_features_df[daily_feature_columns] = (daily_features_df[daily_feature_columns].replace([np.inf, -np.inf], np.nan))

In [80]:
assert daily_features_df["day_of_week"].between(0, 6).all()

assert (daily_features_df["calendar_gap_days"] > 0).all()

valid_positive_fraction = daily_features_df["gap_positive_fraction_20d"].dropna()

assert valid_positive_fraction.between(0, 1).all()

print("Daily feature sanity checks passed.")

Daily feature sanity checks passed.


In [85]:
sample_columns = ["pred_date","symbol","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d","return_1d","return_5d","return_20d","daily_volatility_5d","daily_volatility_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week","actual_return_pct",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",sample_columns,].tail(10)

,pred_date,symbol,lagged_overnight_return_1d,gap_mean_20d,overnight_std_20d,gap_positive_fraction_20d,return_1d,return_5d,return_20d,daily_volatility_5d,daily_volatility_20d,volume_zscore_20d,volume_trend_5d_20d,calendar_gap_days,day_of_week,actual_return_pct
244474,2026-06-15,RELIANCE,1.7247,0.1574,0.7391,0.6000,1.0828,3.4592,-2.1999,1.1672,1.2503,0.1440,1.0137,1.0000,0,0.4897
244475,2026-06-16,RELIANCE,0.4897,0.1909,0.7382,0.6500,1.6679,4.6959,-0.5315,1.2322,1.3117,0.1376,0.9403,1.0000,1,0.3161
244476,2026-06-17,RELIANCE,0.3161,0.1962,0.7387,0.6500,0.2935,5.8707,0.7560,0.8908,1.2930,-0.9885,0.8440,1.0000,2,-0.2026
244477,2026-06-18,RELIANCE,-0.2026,0.2012,0.7355,0.6500,-0.3452,5.1544,-2.3240,1.0784,1.1205,-0.2070,0.8661,1.0000,3,-0.0075
244478,2026-06-19,RELIANCE,-0.0075,0.1732,0.7321,0.6000,-1.4005,1.2761,-2.9713,1.2025,1.1493,0.9741,0.9934,3.0000,4,0.5498
244479,2026-06-22,RELIANCE,0.5498,0.1992,0.7359,0.6000,1.2982,1.4920,-2.0672,1.2425,1.1893,-0.5845,0.9136,1.0000,0,0.1809
244480,2026-06-23,RELIANCE,0.1809,0.1629,0.7168,0.6000,-1.2816,-1.4524,-4.2063,1.1272,1.1919,-0.3257,0.8582,1.0000,1,-0.2902
244481,2026-06-24,RELIANCE,-0.2902,0.1736,0.7078,0.6000,0.3131,-1.4332,-3.1483,1.1298,1.1893,-1.0020,0.8756,1.0000,2,0.3350
244482,2026-06-25,RELIANCE,0.3350,0.2025,0.7017,0.6500,0.3426,-0.7530,-2.3991,1.1617,1.1924,-0.7439,0.8461,4.0000,3,-0.7663
244483,2026-06-29,RELIANCE,-0.7663,0.1142,0.7072,0.6000,-1.2973,-0.6491,-1.5289,1.1345,1.1275,-0.5955,0.7861,1.0000,0,0.4535


In [87]:
daily_features_df[daily_feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
lagged_overnight_return_1d,"301,067.0000",0.2144,1.0074,-34.7406,-0.0672,0.1934,0.5803,63.8095
return_1d,"301,067.0000",0.1246,2.2480,-32.9341,-1.0334,0.0303,1.1607,85.7143
return_5d,"300,235.0000",0.6247,5.1696,-53.8122,-2.2604,0.3411,3.1310,94.9206
return_20d,"297,115.0000",2.4730,10.6418,-72.4351,-3.9055,1.5391,7.6362,139.8804
daily_volatility_20d,"297,115.0000",2.0491,0.9053,0.1686,1.4255,1.8751,2.4575,19.2841
overnight_std_20d,"297,115.0000",0.8305,0.5299,0.0969,0.4893,0.6932,1.0070,14.2957
daily_volatility_5d,"300,235.0000",1.8859,1.2019,0.0004,1.0850,1.6113,2.3709,38.8828
gap_mean_20d,"297,115.0000",0.2130,0.2810,-4.8037,0.0592,0.1974,0.3492,4.5780
gap_positive_fraction_20d,"297,115.0000",0.7103,0.1325,0.0500,0.6000,0.7000,0.8000,1.0000
volume_zscore_20d,"297,115.0000",0.1523,2.2482,-4.6068,-0.6732,-0.3008,0.3350,201.7907


In [89]:
feature_correlation = daily_features_df[daily_feature_columns].corr()

feature_correlation

,lagged_overnight_return_1d,return_1d,return_5d,return_20d,daily_volatility_20d,overnight_std_20d,daily_volatility_5d,gap_mean_20d,gap_positive_fraction_20d,volume_zscore_20d,volume_trend_5d_20d,calendar_gap_days,day_of_week
lagged_overnight_return_1d,1.0000,0.3678,0.2194,0.1247,0.0728,0.0305,0.0747,0.2593,0.1612,0.0341,0.0536,-0.0158,-0.0058
return_1d,0.3678,1.0000,0.4461,0.2278,0.0450,0.0167,0.0825,0.0823,0.0459,0.1518,0.0741,-0.0186,-0.0088
return_5d,0.2194,0.4461,1.0000,0.5048,0.0982,0.0316,0.1541,0.2102,0.1118,0.1019,0.2399,-0.0058,0.0059
return_20d,0.1247,0.2278,0.5048,1.0000,0.2060,0.0597,0.1347,0.4438,0.2361,0.0270,0.0750,-0.0034,0.0070
daily_volatility_20d,0.0728,0.0450,0.0982,0.2060,1.0000,0.6262,0.6298,0.2531,0.0206,0.0194,0.0316,-0.0005,-0.0051
overnight_std_20d,0.0305,0.0167,0.0316,0.0597,0.6262,1.0000,0.3603,0.0856,-0.1663,0.0048,-0.0019,0.0042,-0.0136
daily_volatility_5d,0.0747,0.0825,0.1541,0.1347,0.6298,0.3603,1.0000,0.1340,0.0085,0.1656,0.4322,0.0021,-0.0030
gap_mean_20d,0.2593,0.0823,0.2102,0.4438,0.2531,0.0856,0.1340,1.0000,0.6261,-0.0029,0.0079,-0.0076,0.0108
gap_positive_fraction_20d,0.1612,0.0459,0.1118,0.2361,0.0206,-0.1663,0.0085,0.6261,1.0000,0.0073,0.0141,-0.0117,0.0105
volume_zscore_20d,0.0341,0.1518,0.1019,0.0270,0.0194,0.0048,0.1656,-0.0029,0.0073,1.0000,0.3846,0.0140,0.0203


In [91]:
usable_rows = daily_features_df.dropna(subset=daily_feature_columns)

print("Total rows:", len(daily_features_df))
print("Rows with all daily features available:", len(usable_rows))
print("Coverage: {:.2f}%".format(len(usable_rows) / len(daily_features_df) * 100))

Total rows: 301275
Rows with all daily features available: 297115
Coverage: 98.62%


In [93]:
pd.DataFrame(
{
        "missing": daily_features_df[daily_feature_columns].isna().sum(),
        "missing_pct": daily_features_df[daily_feature_columns].isna().mean() * 100,
    }
).sort_values("missing_pct", ascending=False)

,missing,missing_pct
return_20d,4160,1.3808
daily_volatility_20d,4160,1.3808
overnight_std_20d,4160,1.3808
gap_mean_20d,4160,1.3808
gap_positive_fraction_20d,4160,1.3808
volume_zscore_20d,4160,1.3808
volume_trend_5d_20d,3952,1.3118
return_5d,1040,0.3452
daily_volatility_5d,1040,0.3452
lagged_overnight_return_1d,208,0.0690


In [94]:
daily_features_path = PROCESSED_DATA_DIR / "daily_features_v1.parquet"

daily_features_df.to_parquet(daily_features_path,index=False,)

print("Saved daily feature panel to:")
print(daily_features_path)

Saved daily feature panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_features_v1.parquet


### Minute Feature Engineering

In [99]:
minute_files = sorted(MINUTE_DATA_DIR.glob("*.parquet"))

print("Number of minute parquet files:", len(minute_files))

for file_path in minute_files[:5]: 
    print(file_path.name)

Number of minute parquet files: 208
360ONE.parquet
ABB.parquet
ABCAPITAL.parquet
ADANIENSOL.parquet
ADANIENT.parquet


In [100]:
daily_symbols = {file_path.stem for file_path in daily_files}
minute_symbols = {file_path.stem for file_path in minute_files}

print("Daily symbols:", len(daily_symbols))
print("Minute symbols:", len(minute_symbols))
print("Present only in daily:", sorted(daily_symbols - minute_symbols))
print("Present only in minute:", sorted(minute_symbols - daily_symbols))

Daily symbols: 208
Minute symbols: 208
Present only in daily: []
Present only in minute: []
